# Hybrid Recommenders

Hybrid recommenders combine collaborative signals (what similar users take) with content signals (what the courses are
about). Two approaches, both scored by HitRate@10 and NDCG@10 on the shared leave-one-out enrolment split:

1. **Weighted blend** of an item KNN score and a genre-profile content score, with the weight tuned on validation.
2. **Learning to rank** with LightGBM and XGBoost rankers that learn how to combine seven collaborative, content and
   popularity features.

For every user with at least two enrolments one enrolment is held out as the test item. All hyperparameters, the blend
weight and the rankers' training labels come from a second held-out enrolment (validation), never from the test set.

In [1]:
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd
from lightgbm import LGBMRanker
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBRanker

sys.path.append("..")
from recsys import data, evaluation, metrics, recommenders, results, splits

ratings = data.load_ratings()
index = data.Index(ratings)

## Weighted blend

The blend scores each unseen course as a weighted blend of two signals, each scaled to [0, 1] per user:

- **Collaborative:** item KNN on the enrolment matrix (how often the course is taken by people who took the user's courses).
- **Content:** the user's genre profile (the sum of the genre vectors of the courses they took) dotted with the course's genres.

The blend weight is selected on the validation enrolments. Weight 1 is pure collaborative filtering and weight 0 is pure content.

In [2]:
train_loo, test_loo = splits.leave_one_out(ratings)
fit_loo, val_loo = splits.leave_one_out(train_loo, seed=splits.SEED + 1)

genres = data.load_course_genres().set_index("COURSE_ID").loc[index.items]
genre_features = genres.drop(columns="TITLE").to_numpy(dtype=float)


def hybrid(weight):
    def score(matrix, users):
        similarity = recommenders.item_cosine_similarity(matrix)
        collaborative = recommenders.minmax_rows(recommenders.item_knn_scores(matrix, users, similarity))
        content = recommenders.minmax_rows(recommenders.profile_scores(matrix, users, genre_features))
        return weight * collaborative + (1 - weight) * content

    return score


weights = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
best_weight = evaluation.select_on_validation(hybrid, weights, fit_loo, val_loo, index, "Blend weight")

Blend weight validation NDCG@10 -> 0.0: 0.0529, 0.5: 0.3641, 0.7: 0.4370, 0.8: 0.4558, 0.9: 0.4644, 0.95: 0.4661, 1.0: 0.4665 | selected 1.0


In [3]:
blend_rows = []
for name, weight in [(f"Weighted blend (CF weight {best_weight})", best_weight),
                     ("Blend, content only (weight 0)", 0.0), ("Blend, CF only (weight 1)", 1.0)]:
    scores = evaluation.evaluate_ranking(hybrid(weight), train_loo, test_loo, index)
    blend_rows.append({"family": "Hybrid", "model": name, **scores})

pd.DataFrame(blend_rows).round(4)

,family,model,hit_rate@10,ndcg@10
0,Hybrid,Weighted blend (CF weight 1.0),0.7379,0.5248
1,Hybrid,"Blend, content only (weight 0)",0.1616,0.0626
2,Hybrid,"Blend, CF only (weight 1)",0.7379,0.5248


Validation selects a collaborative weight of 1.0: adding the genre content score does not improve on item KNN at any
weight, so the tuned blend is exactly item KNN. On this dataset, knowing a course's genres adds nothing to knowing who
else took it. The blend is therefore not counted as a hybrid result in the shared comparison.

## Learning-to-rank hybrid with LightGBM and XGBoost

Instead of hand-tuning one blend weight, a gradient boosted ranker learns how to combine many signals. For every user
and every course they have not taken, it gets these features:

- `item_knn`, `user_knn`, `pure_svd`: collaborative filtering scores, with the settings selected in
  `collaborative_filtering.ipynb` (all neighbours for item KNN, k = 200 for user KNN, rank 4 for PureSVD).
- `genre_profile`, `tfidf`: content scores (genre overlap, and TF-IDF similarity of titles and descriptions).
- `log_popularity`: how many training enrolments the course has.
- `user_courses`: how many courses the user has taken.

**Training data without touching the test set.** Features are computed from the inner training enrolments, and the
label is 1 for the user's validation course and 0 for every other candidate. 80% of the validation users train the
ranker and 20% are used for early stopping. For the test set, the same features are recomputed from the full
training enrolments and the ranker scores every candidate.

Both models optimise a LambdaRank / NDCG objective with the courses of each user as one query group.

In [4]:
text = data.load_course_text().set_index("COURSE_ID").loc[index.items]
tfidf = TfidfVectorizer(stop_words="english").fit_transform(text["TITLE"] + " " + text["DESCRIPTION"])
tfidf_similarity = cosine_similarity(tfidf)
np.fill_diagonal(tfidf_similarity, 0.0)

FEATURES = ["item_knn", "user_knn", "pure_svd", "genre_profile", "tfidf", "log_popularity", "user_courses"]


def candidate_features(fit_df, eval_df):
    matrix = splits.interaction_matrix(fit_df, index)
    users, targets = index.encode(eval_df)
    popularity = np.asarray(matrix.sum(axis=0)).ravel()
    user_courses = np.asarray(matrix[users].sum(axis=1)).ravel()
    features = np.stack([
        recommenders.item_knn_scores(matrix, users, recommenders.item_cosine_similarity(matrix)),
        recommenders.user_knn_scores(matrix, users, 200),
        recommenders.pure_svd_scores(matrix, users, 4),
        recommenders.profile_scores(matrix, users, genre_features),
        recommenders.similarity_scores(matrix, users, tfidf_similarity),
        np.tile(np.log1p(popularity), (len(users), 1)),
        np.tile(user_courses[:, None], (1, index.n_items)),
    ], axis=2).astype(np.float32)
    seen = matrix[users].toarray() > 0
    return features, seen, targets


def to_training_rows(features, seen, targets):
    labels = np.zeros(seen.shape, dtype=np.int8)
    labels[np.arange(len(targets)), targets] = 1
    candidates = ~seen
    return features[candidates], labels[candidates], candidates.sum(axis=1)


X_val, seen_val, targets_val = candidate_features(fit_loo, val_loo)
X_test, seen_test, targets_test = candidate_features(train_loo, test_loo)

order = np.random.default_rng(splits.SEED).permutation(len(targets_val))
cut = int(0.8 * len(order))
fit_users, stop_users = np.sort(order[:cut]), np.sort(order[cut:])
X_fit, y_fit, groups_fit = to_training_rows(X_val[fit_users], seen_val[fit_users], targets_val[fit_users])
X_stop, y_stop, groups_stop = to_training_rows(X_val[stop_users], seen_val[stop_users], targets_val[stop_users])
print(f"Ranker training rows: {len(y_fit):,} ({len(groups_fit):,} users) | early stopping rows: {len(y_stop):,}")

Ranker training rows: 2,400,991 (20,159 users) | early stopping rows: 600,259


In [5]:
lgbm_ranker = LGBMRanker(
    objective="lambdarank", n_estimators=1000, learning_rate=0.05, num_leaves=31,
    min_child_samples=50, random_state=splits.SEED, verbose=-1,
)
lgbm_ranker.fit(
    X_fit, y_fit, group=groups_fit, eval_X=(X_stop,), eval_y=(y_stop,), eval_group=[groups_stop],
    eval_at=[10], callbacks=[lgb.early_stopping(50, verbose=False)],
)

xgb_ranker = XGBRanker(
    objective="rank:ndcg", n_estimators=1000, learning_rate=0.05, max_depth=6,
    eval_metric="ndcg@10", early_stopping_rounds=50, random_state=splits.SEED, n_jobs=-1,
)
xgb_ranker.fit(
    X_fit, y_fit, qid=np.repeat(np.arange(len(groups_fit)), groups_fit),
    eval_set=[(X_stop, y_stop)], eval_qid=[np.repeat(np.arange(len(groups_stop)), groups_stop)], verbose=False,
)
print(f"LightGBM stopped at {lgbm_ranker.best_iteration_} trees | XGBoost stopped at {xgb_ranker.best_iteration + 1} trees")

LightGBM stopped at 332 trees | XGBoost stopped at 287 trees


In [6]:
def ranker_scores(model):
    n_users, n_items, n_features = X_test.shape
    return model.predict(X_test.reshape(-1, n_features)).reshape(n_users, n_items)


ranking_rows = [
    {"family": "Hybrid", "model": "LightGBM ranker",
     **metrics.ranking_metrics(ranker_scores(lgbm_ranker), seen_test, targets_test)},
    {"family": "Hybrid", "model": "XGBoost ranker",
     **metrics.ranking_metrics(ranker_scores(xgb_ranker), seen_test, targets_test)},
]

single_signals = [
    {"family": "Single feature", "model": name,
     **metrics.ranking_metrics(X_test[:, :, i], seen_test, targets_test)}
    for i, name in enumerate(FEATURES[:-1])
]
pd.DataFrame(ranking_rows + blend_rows[:1] + single_signals).sort_values("ndcg@10", ascending=False).round(4)

,family,model,hit_rate@10,ndcg@10
1,Hybrid,XGBoost ranker,0.7951,0.5654
0,Hybrid,LightGBM ranker,0.7906,0.5642
4,Single feature,user_knn,0.7856,0.5630
2,Hybrid,Weighted blend (CF weight 1.0),0.7379,0.5248
3,Single feature,item_knn,0.7379,0.5248
5,Single feature,pure_svd,0.6946,0.4712
8,Single feature,log_popularity,0.5728,0.3408
7,Single feature,tfidf,0.3578,0.1762
6,Single feature,genre_profile,0.1616,0.0626


Each single feature is also scored on its own as a ranking, which shows how much the rankers add on top of the
strongest individual signal. Feature importance (share of total gain) shows which signals the rankers rely on:

In [7]:
lgbm_gain = lgbm_ranker.booster_.feature_importance("gain")
xgb_gain = xgb_ranker.get_booster().get_score(importance_type="total_gain")
xgb_gain = np.array([xgb_gain.get(f"f{i}", 0.0) for i in range(len(FEATURES))])
pd.DataFrame(
    {"LightGBM": lgbm_gain / lgbm_gain.sum(), "XGBoost": xgb_gain / xgb_gain.sum()}, index=FEATURES
).sort_values("LightGBM", ascending=False).round(3)

,LightGBM,XGBoost
user_knn,0.866,0.874
log_popularity,0.034,0.034
user_courses,0.033,0.034
pure_svd,0.031,0.031
item_knn,0.016,0.011
tfidf,0.013,0.011
genre_profile,0.006,0.004


Both rankers beat every single signal, but only narrowly: the gain over the best single model (user KNN) is small,
and almost all of the learned weight goes to collaborative signals and popularity. The two content features together
account for only a few percent of the total gain. The rankers are hybrids in their inputs, but on this dataset the
content signal contributes little, which matches the weighted blend above.

In [8]:
results.save(ranking_rows, "ranking", source="hybrid_recommenders");